# **Test pipeline**

## Setup environment

In [ ]:
import sys
import os
import getpass
import pandas as pd
from pathlib import Path

sys.path.append(os.path.abspath(".."))

# Autenticación de Hugging Face
os.environ["HF_TOKEN"] = getpass.getpass("Pega tu token de HuggingFace: ")

In [ ]:
from src.utils.config_loader import load_config

# Cargar parámetros generales del proyecto
config = load_config("../configs/config.yaml")

print(f"Proyecto: {config['project']['name']}")
print(f"GPUs configuradas: {config['hardware']['num_gpus']}")
print(f"Directorio de entrada: {config['paths']['raw_audio_dir']}")

## Batch run

In [ ]:
from src.pipeline.batch_runner import BatchRunner

# Configurar rutas 
input_directory = "../data/agora/c003/" 
output_directory = "../data/processed/"

runner = BatchRunner(
    input_dir=input_directory,
    output_dir=output_directory,
    num_gpus=config["hardware"]["num_gpus"]
)

# Iniciar procesamiento en paralelo
resultados_batch = runner.run()

# Mostrar resumen del estado de los archivos
df_status = pd.DataFrame(resultados_batch)
display(df_status)

## Metrics

In [ ]:
from src.metrics.calculator import MetricsCalculator

processed_dir = Path(output_directory)
csv_files = list(processed_dir.glob("*_transcript_speakers.csv"))

if not csv_files:
    print("No se encontraron resultados procesados.")
else:
    archivo_prueba = csv_files[0]
    print(f"Calculando métricas para: {archivo_prueba.name}\n")
    
    df_resultados = pd.read_csv(archivo_prueba)
    
    tiempos = MetricsCalculator.calculate_speaking_time(df_resultados)
    print(f"Distribución de tiempo de habla:")
    print(f" - Mujeres: {tiempos['female_pct']:.1f}%")
    print(f" - Hombres: {tiempos['male_pct']:.1f}%\n")
    
    umbral = config["business_rules"]["overlap_threshold_seconds"]
    df_interrupciones = MetricsCalculator.detect_interruptions(df_resultados, overlap_threshold=umbral)
    
    print(f"Total de interrupciones detectadas (>={umbral}s): {len(df_interrupciones)}")
    
    if not df_interrupciones.empty:
        # Filtrar interrupciones específicas de hombre a mujer
        h_a_m = df_interrupciones[df_interrupciones["type"] == "female_to_male"]
        print(f"Interrupciones de hombre a mujer: {len(h_a_m)}")
        display(df_interrupciones.head())